# A1 → A2 → A3: hậu xử lý M1/M2 DiffMul trên Kaggle

- **A1**: calibration (temperature / Platt / isotonic) + chọn ngưỡng 5-fold OOF trên validation.
- **A2**: SWA-lite (trung bình best + last checkpoint), chọn biến thể bằng validation.
- **A3**: ensemble M1 + M2, quét trọng số trên OOF validation.

Checkpoint được gắn từ kernel `bcb-tuning-m1-m2-diffmul-paired`; parquet validation/test được gắn từ dataset.
Mọi ngưỡng/trọng số chốt trên validation — test chỉ đánh giá cuối kèm bootstrap CI + McNemar.


In [ ]:
# 1. SETUP + STREAM LOG
import base64, io, json, os, shutil, subprocess, sys, tarfile, time
from pathlib import Path

def sh(cmd, **kw):
    print(f'\n$ {cmd}', flush=True)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    log_file = Path('/kaggle/working/live_progress.log')
    with log_file.open('a', encoding='utf-8') as lf:
        for line in iter(proc.stdout.readline, ''):
            print(line, end='', flush=True)
            lf.write(line)
            lf.flush()
    proc.stdout.close()
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f'Lệnh thất bại (rc={rc}): {cmd}')
    return ''

print('Python:', sys.version.split()[0], flush=True)
print('CPU count:', os.cpu_count(), flush=True)

In [ ]:
# 2. GIẢI NÉN MÃ NGUỒN
payload = '...'
repo_root = Path('/kaggle/working/project')
repo_root.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(payload)), mode='r:gz') as tar:
    tar.extractall(repo_root)
os.chdir(repo_root)
print('Repo root:', os.getcwd(), flush=True)
sh('ls -la src/postprocess scripts/postprocess_diffmul.py')

In [ ]:
# 3. GPU + DEPENDENCIES
import torch
print('CUDA:', torch.cuda.is_available(), flush=True)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0), flush=True)
    print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), flush=True)
else:
    raise SystemError('Kernel chưa bật GPU!')
sh('pip install -q -U transformers datasets scikit-learn scipy pyarrow')

In [ ]:
# 4. LIÊN KẾT PARQUET VAL/TEST VÀ CHECKPOINT TỪ KERNEL SOURCE
input_path = Path('/kaggle/input')
print('Input dirs:', [p.name for p in input_path.iterdir()], flush=True)

data_raw = Path('data/raw/bigclonebench')
data_raw.mkdir(parents=True, exist_ok=True)
for name in ('bcb_validation.parquet', 'bcb_test.parquet'):
    candidates = list(input_path.rglob(name))
    dest = data_raw / name
    if candidates and not dest.exists():
        sh(f'cp "{candidates[0]}" "{dest}"')
    print(f'{name}: exists={dest.exists()}, size={dest.stat().st_size/1e6:.1f} MB' if dest.exists() else f'{name}: MISSING', flush=True)

# Dataset Kaggle không chứa parquet -> tải lại đúng split đã dùng lúc train
if any(not (data_raw / n).exists() for n in ('bcb_validation.parquet', 'bcb_test.parquet')):
    print('Parquet chưa có trong input -> tải từ Hugging Face...', flush=True)
    sh('python scripts/download_bcb_subsets.py --splits-custom --train-samples 30000 --val-samples 3000 --test-samples 5000')

assert (data_raw / 'bcb_validation.parquet').exists(), 'Thiếu bcb_validation.parquet!'
assert (data_raw / 'bcb_test.parquet').exists(), 'Thiếu bcb_test.parquet!'

# Copy checkpoint từ dataset chuyên dụng (đã upload)
dataset_ckpts = {
    'm1_best_checkpoint.pt': 'results/exp_m1_diffmul/best_checkpoint.pt',
    'm2_best_checkpoint.pt': 'results/exp_m2_diffmul/best_checkpoint.pt',
    'm2_last_checkpoint.pt': 'results/exp_m2_diffmul/last_checkpoint.pt',
}
for f in input_path.rglob('*.pt'):
    if f.name in dataset_ckpts:
        dest = dataset_ckpts[f.name]
        Path(dest).parent.mkdir(parents=True, exist_ok=True)
        if not Path(dest).exists():
            shutil.copy2(f, dest)
        print(f'Dataset ckpt: {f.name} -> {dest}', flush=True)

# Fallback: bundle/output kernel cũ (nếu dataset chưa có)
for tar_f in input_path.rglob('*.tar.gz'):
    low = tar_f.name.lower()
    if 'diff' in low or 'tuning' in low or 'bundle' in low:
        print(f'Giải nén {tar_f} ...', flush=True)
        sh(f'tar -xzf "{tar_f}" -C /kaggle/working/')
for f in list(input_path.rglob('*.pt')) + list(Path('/kaggle/working').glob('exp_*_checkpoint.pt')):
    low = str(f).lower()
    dest = None
    if 'exp_m1_diffmul_best' in low: dest = 'results/exp_m1_diffmul/best_checkpoint.pt'
    elif 'exp_m2_diffmul_best' in low: dest = 'results/exp_m2_diffmul/best_checkpoint.pt'
    elif 'exp_m2_diffmul_last' in low: dest = 'results/exp_m2_diffmul/last_checkpoint.pt'
    if dest and not Path(dest).exists():
        Path(dest).parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, dest)

# M1 last không cần: best epoch 8 == last epoch 8, pipeline tự bỏ qua SWA.
required = list(dataset_ckpts.values())
for dest in required:
    p = Path(dest)
    print(f'{dest}: exists={p.exists()}' + (f', {p.stat().st_size/1e6:.1f} MB' if p.exists() else ''), flush=True)
    assert p.exists(), f'Thiếu checkpoint {dest}'

In [ ]:
# 5. CHẠY A1 -> A2 -> A3
print('=' * 70)
print('>>> BAT DAU POSTPROCESS A1 -> A2 -> A3 <<<')
print('=' * 70)
cmd = (
    'python scripts/postprocess_diffmul.py '
    '--val-file data/raw/bigclonebench/bcb_validation.parquet '
    '--test-file data/raw/bigclonebench/bcb_test.parquet '
    '--out-dir results/exp_diffmul_postprocess '
    '--batch-size 32 --precision fp16 --n-folds 5 --seed 42'
)
t0 = time.perf_counter()
sh(cmd)
print(f'>>> HOAN TAT TRONG {(time.perf_counter()-t0)/60:.1f} PHUT!', flush=True)
print(Path('results/exp_diffmul_postprocess/postprocess_report.md').read_text(encoding='utf-8'), flush=True)

In [ ]:
# 6. THU THẬP KẾT QUẢ + AUTO-CLEANUP
WORK = Path('/kaggle/working')
CLEAN = WORK / 'clean_postprocess'
CLEAN.mkdir(parents=True, exist_ok=True)
out_dir = repo_root / 'results' / 'exp_diffmul_postprocess'

for p in sorted(out_dir.rglob('*')):
    if p.is_file() and p.suffix in ('.json', '.md', '.npy', '.log'):
        name = f'postprocess_{p.name}'
        shutil.copy2(p, CLEAN / name)
        shutil.copy2(p, WORK / name)
        print(f'  [SAVED] {name}', flush=True)

log_file = WORK / 'live_progress.log'
if log_file.exists():
    shutil.copy2(log_file, CLEAN / 'postprocess_live_progress.log')
    shutil.copy2(log_file, WORK / 'postprocess_live_progress.log')

bundle = WORK / 'diffmul_postprocess_bundle.tar.gz'
with tarfile.open(bundle, 'w:gz') as tar:
    for f in CLEAN.glob('*'):
        tar.add(f, arcname=f.name)
print(f'>>> Bundle: {bundle} ({bundle.stat().st_size/1e6:.1f} MB)', flush=True)

os.chdir(WORK)
for junk in ('project', 'data', 'artifacts', 'clean_postprocess', 'ckpt_dl'):
    shutil.rmtree(WORK / junk, ignore_errors=True)
# Xóa checkpoint/bundle tải về để không upload lại hàng GB khi kernel kết thúc
for pattern in ('exp_m1_diffmul_*', 'exp_m2_diffmul_*', 'diff_mul_tuning_bundle.tar.gz'):
    for p in WORK.glob(pattern):
        if p.is_file():
            p.unlink()
print('DANH SÁCH OUTPUT:', flush=True)
for p in sorted(WORK.glob('postprocess_*')):
    print(f'  {p.name} ({p.stat().st_size/1e6:.2f} MB)', flush=True)
print('>>> XONG 100%! <<<', flush=True)